In [1]:
from pathlib import Path


import pandas as pd
import xarray as xr
from imagematerials.util import dataset_to_array
from imagematerials.vehicles.constants import maintenance_lifetime_per_mode

import matplotlib.pyplot as plt

import warnings
from pathlib import Path

import prism

from imagematerials.concepts import knowledge_graph
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.util import export_to_netcdf, import_from_netcdf, rebroadcast_prep_data
from imagematerials.vehicles import (
    preprocess,
)

In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema.nc")

if not prep_fp.is_file():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        orig_prep_data = preprocess(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

In [3]:


# Copy dimensiomns from material_fractions for xr_maintenance_material
materials = prep_data['material_fractions'].coords["material"]
types = prep_data['material_fractions'].coords["Type"]

maintenance_material_pd : pd.DataFrame = pd.read_csv(
    "../data/raw/vehicles/standard_data/all_vehicle_maintenance_image.csv", index_col=0)

maintenance_material_pd['Li'] = 0
maintenance_material_pd['Mn'] = 0
maintenance_material_pd['Ni'] = 0
maintenance_material_pd['Ti'] = 0

stacked_maintenance_material = maintenance_material_pd.set_index("Type").stack().rename_axis(index=["Type", "material"]).reset_index(name="value")

stacked_maintenance_material = stacked_maintenance_material.set_index(["Type", "material"])

stacked_maintenance_material_xr = stacked_maintenance_material.to_xarray()
maintenance_material = dataset_to_array(stacked_maintenance_material_xr, ["Type", "material"], [])

modes = list(maintenance_material.coords['Type'].values)
expected_lifetimes = xr.DataArray(
    data=[maintenance_lifetime_per_mode[mode] for mode in modes],
    dims=["Type"],
    coords={"Type": modes},
    name="vehicle_lifetime"
)

maintenance_material_per_year = (maintenance_material / expected_lifetimes)

maintenance_material_per_year_broadcasted = vehicle_knowledge_graph.rebroadcast_xarray_impute(
    maintenance_material_per_year, types.values)

NameError: name 'vehicle_knowledge_graph' is not defined

In [ ]:
types.values

In [ ]:
maintenance_material_per_year_broadcasted

In [4]:
prep_data["maintenance_material_fractions"] 

<xarray.DataArray (Type: 8, material: 14)> Size: 896B
array([[7.61218260e-03, 6.21896191e-04, 1.05035714e-04, 8.44285714e-05,
        0.00000000e+00, 0.00000000e+00, 9.85714286e-06, 0.00000000e+00,
        1.92847837e-03, 9.71081567e-04, 5.49011364e-03, 8.95075717e-03,
        0.00000000e+00, 0.00000000e+00],
       [1.27584872e-03, 0.00000000e+00, 5.27840909e-04, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        4.04829545e-04, 3.44744318e-04, 0.00000000e+00, 1.22419070e-02,
        0.00000000e+00, 0.00000000e+00],
       [1.16725438e-02, 0.00000000e+00, 2.00100751e-03, 1.54142582e-07,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 8.46570328e-05, 5.01709056e-04, 1.96765739e-02,
        0.00000000e+00, 3.27552987e-08],
       [0.00000000e+00, 0.00000000e+00, 2.14285714e-05, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        9.28571429e-04, 8.57142857e-04, 1.66428571e-02, 1.57142857e-03,
        0.00000000e+00, 0.00000000e+00],
       [2.43312500e-03, 0.00000000e+00, 7.99062500e-04, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        4.68750000e-04, 4.03125000e-04, 0.00000000e+00, 2.33146875e-02,
        0.00000000e+00, 0.00000000e+00],
       [2.94535809e-04, 0.00000000e+00, 1.04482759e-04, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 5.72944297e-05, 0.00000000e+00, 3.34899204e-03,
        0.00000000e+00, 0.00000000e+00],
       [2.48037135e-04, 0.00000000e+00, 9.91114058e-05, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 5.72944297e-05, 0.00000000e+00, 2.93587533e-03,
        0.00000000e+00, 0.00000000e+00],
       [3.47467025e-03, 0.00000000e+00, 5.95657758e-04, 3.20417850e-04,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 1.35942046e-03, 4.53751666e-04, 5.85730129e-03,
        0.00000000e+00, 2.31157795e-04]])
Coordinates:
  * Type      (Type) <U25 800B 'Cars' 'Heavy Freight Trucks' ... 'Trains'
  * material  (material) <U9 504B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Ti' 'Wood'

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=True, 
    material=material)

In [ ]:
main_model_normal.simulate(simulation_timeline)

In [ ]:
# Sum general inflow over Region
inflow_total = main_model_normal.material_model.inflow_materials.to_array().sum(dim="Region")

# Sum maintenance inflow over Region and materials
inflow_maintenance = main_model_normal.maintenance_model.inflow_maintenance.to_array().sum(dim=["Region"])#.sel(Type="Trains")

# Convert to pandas Series
inflow_total_pd = inflow_total.to_series()
inflow_maintenance_pd = inflow_maintenance.to_series()

# Combine into a DataFrame
df_plot = pd.DataFrame({
    "Total Inflow": inflow_total_pd,
    "Maintenance Inflow": inflow_maintenance_pd
})

df_plot

In [ ]:
# Sum general inflow over Region
production_df = main_model_normal.material_model.inflow_materials.to_array().sum(dim=["Region", "Type"]).to_pandas()

# Sum maintenance inflow over Region and materials
maintenance_df = main_model_normal.maintenance_model.inflow_maintenance.to_array().sum(dim=["Region", "Type"]).to_pandas()

# Filter out zero-only materials
nonzero_materials = (
    (maintenance_df != 0).any(axis=0) |
    (production_df != 0).any(axis=0) 
)

maintenance_df = maintenance_df.loc[:, nonzero_materials]
production_df = production_df.loc[:, nonzero_materials]

# Filter from 1972 onwards
maintenance_df = maintenance_df[maintenance_df.index >= 1972]
production_df = production_df[production_df.index >= 1972]

# Total maintenance sum (used for offset and line)
maintenance_total = maintenance_df.sum(axis=1)

production_df_smooth = production_df.rolling(window=5, center=True).mean()

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Plot maintenance materials as stacked area
production_df_smooth.plot.area(ax=ax, stacked=True, colormap="tab20", alpha=0.7)

# Plot a thick line on top of maintenance total
ax.plot(maintenance_total.index, maintenance_total, color="black", linewidth=3, label="Total Maintenance")


# Final touches
ax.set_xlabel("Year")
ax.set_ylabel("Material Flow (Mt)")
ax.set_title("Maintenance and Production Material Flows")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Materials")
plt.tight_layout()
plt.show()

